# Google QEC — Silver stage

Builds the two required Silver tables for `google_qec` from the Bronze zip, plus the trace and data-issue rows they need:

| Output | One row is... |
| --- | --- |
| `experiment` (`silver/google_qec/experiment.parquet`) | one experiment directory (5 rows) |
| `shot` (`silver/google_qec/shot.parquet`) | one aligned hardware shot (5 x 50,000 rows) |
| `trace` rows for `results/part1/source_trace.parquet` | one Bronze member a Silver row was read from (a shot has 8) |
| `issues` rows for `results/part1/data_issues.parquet` | one rejected record |

**Failure policy** (from `assignment/brief.md`): a problem that makes a whole experiment unusable (bad metadata, name vs `properties.yml` mismatch, wrong file length, non-binary or misaligned `01` file) **raises `GoogleSilverError` and stops the run**. A problem confined to one shot (non-zero padding bits) **excludes that shot** and writes a data-issue row.

**Proposed IDs (agree with the team, P5 owns tracing):** `experiment_id` = the experiment directory name; `source_record_id` = `stable_record_hash` of the source + directory + shot index (experiments: of the `properties.yml` member). Neither depends on run time or row order.

Nothing here writes files. The write location (MinIO `silver/` or local) is still a team decision. When the functions are stable, move cells 3-7 into `prepare_data.py` (or a Google module it calls).

## 1. Setup and load the Google Bronze archive

In [1]:
import hashlib
import io
import zipfile
from dataclasses import dataclass

import pyarrow as pa
import pyarrow.compute as pc
import yaml

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import bronze_inventory, minio_client
from quantum_lake_student.formats import b8_record_bytes, iter_b8_records, parse_01_records
from quantum_lake_student.models import Severity, stable_record_hash

settings = Settings.from_environment()
google_object = next(name for name in dict(bronze_inventory(settings)) if "source=google_qec" in name)

response = minio_client(settings).get_object(settings.s3_bucket, google_object)
try:
    google_bytes = response.read()
finally:
    response.close()
    response.release_conn()

google_sha256 = hashlib.sha256(google_bytes).hexdigest()
zf = zipfile.ZipFile(io.BytesIO(google_bytes))
print(google_object, f"{len(google_bytes):,} bytes", google_sha256[:12] + "...")

bronze/source=google_qec/google-surface-code-curated.zip 14,638,673 bytes 5d6a24f89f05...


## 2. Constants and target schemas (types come from `assignment/silver-tables.md`)

In [2]:
import re

SOURCE_NAME = "google_qec"
DECODERS = ("belief_matching", "correlated_matching", "pymatching", "tensor_network_contraction")
REQUIRED_PROPERTIES = (
    "basis", "distance", "rounds", "shots",
    "center_data_qubit_row", "center_data_qubit_col",
    "circuit_measurements", "circuit_sweep_bits", "circuit_detectors",
)
DIR_PATTERN = re.compile(
    r"^surface_code_b(?P<basis>[A-Za-z])_d(?P<distance>\d+)_r(?P<rounds>\d+)_center_(?P<row>\d+)_(?P<col>\d+)$"
)

# (shot-table column, companion file, properties.yml key holding the bits per shot)
B8_FILES = (
    ("measurement_bits", "measurements.b8", "circuit_measurements"),
    ("sweep_bits", "sweep.b8", "circuit_sweep_bits"),
    ("detector_bits", "detection_events.b8", "circuit_detectors"),
)

EXPERIMENT_SCHEMA = pa.schema([
    ("source_record_id", pa.string()), ("experiment_id", pa.string()), ("basis", pa.string()),
    ("distance", pa.int32()), ("rounds", pa.int32()), ("shots", pa.int64()),
    ("center_row", pa.int32()), ("center_col", pa.int32()),
    ("measurement_count", pa.int32()), ("detector_count", pa.int32()),
])
SHOT_SCHEMA = pa.schema([
    ("source_record_id", pa.string()), ("experiment_id", pa.string()), ("shot_index", pa.int64()),
    ("measurement_bits", pa.binary()), ("sweep_bits", pa.binary()), ("detector_bits", pa.binary()),
    ("detector_event_count", pa.int32()), ("actual_observable_flip", pa.bool_()),
    *[(f"{name}_prediction", pa.bool_()) for name in DECODERS],
])
TRACE_SCHEMA = pa.schema([
    ("source_record_id", pa.string()), ("source_name", pa.string()), ("bronze_object", pa.string()),
    ("archive_member", pa.string()), ("record_locator", pa.string()), ("input_sha256", pa.string()),
])


class GoogleSilverError(RuntimeError):
    """A check failed that makes an experiment unusable. The caller must stop the run."""

## 3. Check functions

Each check either returns quietly, returns the offending shots, or raises `GoogleSilverError`.

In [4]:
def experiment_dirs(members):
    """Experiment directories in the archive, identified by their properties.yml."""
    return sorted({m.rsplit("/", 1)[0] for m in members if m.endswith("/properties.yml")})


def read_properties(zf, exp_dir):
    props = yaml.safe_load(zf.read(f"{exp_dir}/properties.yml"))
    missing = [k for k in REQUIRED_PROPERTIES if not isinstance(props, dict) or k not in props]
    if missing:
        raise GoogleSilverError(f"{exp_dir}/properties.yml is missing key(s): {missing}")
    for key in REQUIRED_PROPERTIES:
        if key != "basis" and (not isinstance(props[key], int) or props[key] < 0):
            raise GoogleSilverError(f"{exp_dir}: {key}={props[key]!r} is not a non-negative integer")
    if props["shots"] <= 0:
        raise GoogleSilverError(f"{exp_dir}: shots must be positive, got {props['shots']}")
    return props


def check_experiment_name(exp_dir, props):
    """The directory name and properties.yml must describe the same experiment."""
    match = DIR_PATTERN.match(exp_dir)
    if not match:
        raise GoogleSilverError(f"experiment directory name {exp_dir!r} does not match the documented pattern")
    from_name = (match["basis"], int(match["distance"]), int(match["rounds"]), int(match["row"]), int(match["col"]))
    from_props = (props["basis"], props["distance"], props["rounds"],
                  props["center_data_qubit_row"], props["center_data_qubit_col"])
    if from_name != from_props:
        raise GoogleSilverError(
            f"{exp_dir}: name says (basis, distance, rounds, row, col)={from_name} "
            f"but properties.yml says {from_props}"
        )


def check_b8_size(data, bits_per_shot, shots, label):
    """Return bytes per shot; raise unless the file is exactly shots x that many bytes."""
    record_bytes = 0 if bits_per_shot == 0 else b8_record_bytes(bits_per_shot)
    if len(data) != record_bytes * shots:
        raise GoogleSilverError(
            f"{label}: {len(data)} bytes, expected {record_bytes * shots} "
            f"({shots} shots x {record_bytes} bytes for {bits_per_shot} bits)"
        )
    return record_bytes


def dirty_padding_shots(data, bits_per_shot):
    """Shot indices whose unused padding bits (high bits of the last byte) are not all zero."""
    pad_start = bits_per_shot % 8
    if bits_per_shot == 0 or pad_start == 0:
        return []
    record_bytes = b8_record_bytes(bits_per_shot)
    mask = (0xFF << pad_start) & 0xFF
    return [i for i, last in enumerate(data[record_bytes - 1 :: record_bytes]) if last & mask]


def read_01_flags(data, shots, label):
    """Parse a Stim 01 file: one 0/1 per line, exactly one line per shot."""
    try:
        values = parse_01_records(data)
    except ValueError as error:
        raise GoogleSilverError(f"{label}: {error}") from error
    if len(values) != shots:
        raise GoogleSilverError(f"{label}: {len(values)} lines, expected {shots}")
    return [bool(v) for v in values]

## 4. Identifiers and the experiment row

In [5]:
def experiment_source_record_id(exp_dir):
    return stable_record_hash({"source": SOURCE_NAME, "member": f"{exp_dir}/properties.yml"})


def shot_source_record_id(exp_dir, shot_index):
    return stable_record_hash({"source": SOURCE_NAME, "experiment_id": exp_dir, "shot_index": shot_index})


def build_experiment_row(exp_dir, props):
    return {
        "source_record_id": experiment_source_record_id(exp_dir),
        "experiment_id": exp_dir,
        "basis": props["basis"],
        "distance": props["distance"],
        "rounds": props["rounds"],
        "shots": props["shots"],
        "center_row": props["center_data_qubit_row"],
        "center_col": props["center_data_qubit_col"],
        "measurement_count": props["circuit_measurements"],
        "detector_count": props["circuit_detectors"],
    }

## 5. Assemble the aligned shots of one experiment

Every companion file is validated first. A shot is only published when its bytes in all three `b8` files and its line in all five `01` files are valid. `measurement_bits`, `sweep_bits` and `detector_bits` are the **packed `b8` rows, unchanged**; `detector_event_count` is the number of set bits in `detector_bits`.

In [6]:
def _padding_issue(run_id, exp_dir, shot_index, member):
    source_record_id = shot_source_record_id(exp_dir, shot_index)
    rule_id = "google.b8_padding_bits_zero"
    return {
        "issue_id": stable_record_hash([rule_id, source_record_id, member]),
        "run_id": run_id,
        "source_record_id": source_record_id,
        "rule_id": rule_id,
        "severity": Severity.ERROR.value,
        "observed_value": f"{exp_dir}/{member} shot_index={shot_index}",
        "action": "excluded_from_silver",
        "reason": f"unused padding bits in a {member} record must be zero",
    }


def build_shot_columns(zf, exp_dir, props, run_id):
    """Validate every companion file of one experiment; return (column lists, issue rows)."""
    shots = props["shots"]
    packed = {}
    bad_shots = {}
    for column, filename, key in B8_FILES:
        data = zf.read(f"{exp_dir}/{filename}")
        bits = props[key]
        record_bytes = check_b8_size(data, bits, shots, f"{exp_dir}/{filename}")
        packed[column] = (data, record_bytes)
        for index in dirty_padding_shots(data, bits):
            bad_shots.setdefault(index, filename)

    flag_path = f"{exp_dir}/obs_flips_actual.01"
    actual = read_01_flags(zf.read(flag_path), shots, flag_path)
    predictions = {}
    for name in DECODERS:
        flag_path = f"{exp_dir}/obs_flips_predicted_by_{name}.01"
        predictions[name] = read_01_flags(zf.read(flag_path), shots, flag_path)

    columns = {field.name: [] for field in SHOT_SCHEMA}
    issues = []
    for index in range(shots):
        if index in bad_shots:
            issues.append(_padding_issue(run_id, exp_dir, index, bad_shots[index]))
            continue
        row = {column: data[index * rb : (index + 1) * rb] for column, (data, rb) in packed.items()}
        columns["source_record_id"].append(shot_source_record_id(exp_dir, index))
        columns["experiment_id"].append(exp_dir)
        columns["shot_index"].append(index)
        columns["measurement_bits"].append(row["measurement_bits"])
        columns["sweep_bits"].append(row["sweep_bits"])
        columns["detector_bits"].append(row["detector_bits"])
        columns["detector_event_count"].append(int.from_bytes(row["detector_bits"], "little").bit_count())
        columns["actual_observable_flip"].append(actual[index])
        for name in DECODERS:
            columns[f"{name}_prediction"].append(predictions[name][index])
    return columns, issues

## 6. Trace rows and the whole-archive builder

A shot is assembled from 8 Bronze members, so it gets 8 trace rows that share one `source_record_id`. `b8` locators are `shot_index=<n>`; `01` locators are `line=<n+1>`.

In [7]:
def _add_trace(trace, source_record_id, bronze_object, member, locator, input_sha256):
    trace["source_record_id"].append(source_record_id)
    trace["source_name"].append(SOURCE_NAME)
    trace["bronze_object"].append(bronze_object)
    trace["archive_member"].append(member)
    trace["record_locator"].append(locator)
    trace["input_sha256"].append(input_sha256)


@dataclass
class GoogleSilver:
    experiment: pa.Table
    shot: pa.Table
    trace: pa.Table
    issues: list
    shots_read: int


def build_google_silver(zf, *, bronze_object, input_sha256, run_id):
    """Build experiment table, shot table, trace rows and issue rows for the whole archive."""
    experiment_rows, shot_tables, issues = [], [], []
    trace = {name: [] for name in TRACE_SCHEMA.names}
    shots_read = 0

    dirs = experiment_dirs(zf.namelist())
    if not dirs:
        raise GoogleSilverError("no experiment directories (properties.yml) found in the archive")
    for exp_dir in dirs:
        props = read_properties(zf, exp_dir)
        check_experiment_name(exp_dir, props)
        row = build_experiment_row(exp_dir, props)
        experiment_rows.append(row)
        _add_trace(trace, row["source_record_id"], bronze_object, f"{exp_dir}/properties.yml",
                   "properties.yml", input_sha256)

        columns, exp_issues = build_shot_columns(zf, exp_dir, props, run_id)
        shots_read += props["shots"]
        issues += exp_issues
        shot_tables.append(pa.table(columns, schema=SHOT_SCHEMA))

        b8_members = [f"{exp_dir}/{filename}" for _c, filename, _k in B8_FILES]
        flag_members = [f"{exp_dir}/obs_flips_actual.01"] + [
            f"{exp_dir}/obs_flips_predicted_by_{name}.01" for name in DECODERS
        ]
        for source_record_id, index in zip(columns["source_record_id"], columns["shot_index"]):
            shot_locator, line_locator = f"shot_index={index}", f"line={index + 1}"
            for member in b8_members:
                _add_trace(trace, source_record_id, bronze_object, member, shot_locator, input_sha256)
            for member in flag_members:
                _add_trace(trace, source_record_id, bronze_object, member, line_locator, input_sha256)

    return GoogleSilver(
        experiment=pa.Table.from_pylist(experiment_rows, schema=EXPERIMENT_SCHEMA),
        shot=pa.concat_tables(shot_tables),
        trace=pa.table(trace, schema=TRACE_SCHEMA),
        issues=issues,
        shots_read=shots_read,
    )

## 7. Run it on the real Bronze archive (about a minute)

In [8]:
silver = build_google_silver(zf, bronze_object=google_object, input_sha256=google_sha256, run_id="notebook-exploration")
print("experiment rows:", silver.experiment.num_rows)
print("shot rows:      ", f"{silver.shot.num_rows:,}")
print("trace rows:     ", f"{silver.trace.num_rows:,}")
print("issues:         ", len(silver.issues))
silver.experiment.to_pandas()

experiment rows: 5
shot rows:       250,000
trace rows:      2,000,005
issues:          0


,source_record_id,experiment_id,basis,distance,rounds,shots,center_row,center_col,measurement_count,detector_count
0,ac09c968bce728cd35ec15cc0f9411b221bcff09392f2b...,surface_code_bX_d3_r25_center_3_5,X,3,25,50000,3,5,209,200
1,28498044b78d049865e6dc9f3a1df14a7d2168defc8850...,surface_code_bX_d3_r25_center_5_3,X,3,25,50000,5,3,209,200
2,618c00fc9c27cc274d344c0f98a5bce13c481955cb7bea...,surface_code_bX_d3_r25_center_5_7,X,3,25,50000,5,7,209,200
3,c63864dfec686a9daa5bc51327303fad9ae0bc0d224463...,surface_code_bX_d3_r25_center_7_5,X,3,25,50000,7,5,209,200
4,42fad5a6b21925060907b67fcffabb7434937b6d9bcf20...,surface_code_bX_d5_r25_center_5_5,X,5,25,50000,5,5,625,600


In [9]:
silver.shot.slice(0, 3).to_pandas()

,source_record_id,experiment_id,shot_index,measurement_bits,sweep_bits,detector_bits,detector_event_count,actual_observable_flip,belief_matching_prediction,correlated_matching_prediction,pymatching_prediction,tensor_network_contraction_prediction
0,f9dbd64dc0d6358d08e4f170176a7978cdc890e1cc9729...,surface_code_bX_d3_r25_center_3_5,0,b'\x85 \xad\x905\x90_\xb8\x1d\xb8\x1d\xb8]\xf8...,b'\xd0\x00',b'\x00\x80\x82\t\x00\xa0&\x04\x00\x00\x00\x00\...,27,True,True,True,True,True
1,e905837969e078d4a2245bea4659ebdd37a6a628479610...,surface_code_bX_d3_r25_center_3_5,1,b'\xac\x02\xaf<\x9dp\xcd\xb8\x9dx\xddy\xbcY\xf...,b'\xd0\x00',b'\xb3\x80`C\x80\x84\x01\r\x08\x04\x10\x00\x06...,36,True,True,False,False,True
2,7fb3d12c4ccd5dd2bb231b263e8f3c234a4aae74079e80...,surface_code_bX_d3_r25_center_3_5,2,b'\xd3\xf4S\xb6S\xb6\x11\xa6\x13\xe6\xc1l\xfb\...,b'\xd0\x00',b'$(\x00\x04\x04$ \x01\x01%\x88 \xa3\x04\x01\x...,32,True,True,True,False,True


## 8. Reconciliation and self-checks

The brief requires *rows read = rows accepted + rows sent to `data_issues`*, stable identifiers, and trace coverage.

In [10]:
accepted, rejected = silver.shot.num_rows, len(silver.issues)
print(f"shots read {silver.shots_read:,} = accepted {accepted:,} + rejected {rejected:,}:", silver.shots_read == accepted + rejected)
print("experiment.shots sums to shots read:", pc.sum(silver.experiment["shots"]).as_py() == silver.shots_read)

assert silver.experiment.schema == EXPERIMENT_SCHEMA and silver.shot.schema == SHOT_SCHEMA and silver.trace.schema == TRACE_SCHEMA
assert pc.count_distinct(silver.shot["source_record_id"]).as_py() == silver.shot.num_rows, "shot ids must be unique"
assert pc.count_distinct(silver.experiment["source_record_id"]).as_py() == silver.experiment.num_rows

# trace coverage: every Silver source_record_id appears in the trace, shots with exactly 8 rows
traced = set(silver.trace["source_record_id"].to_pylist())
silver_ids = set(silver.shot["source_record_id"].to_pylist()) | set(silver.experiment["source_record_id"].to_pylist())
print("trace covers every Silver row:", silver_ids <= traced)
print("trace rows per shot:", silver.trace.num_rows - silver.experiment.num_rows, "==", 8 * silver.shot.num_rows)

# bit order: recount the set bits of shot 0's detector row one bit at a time
first = silver.shot.slice(0, 1).to_pylist()[0]
exp_row = next(r for r in silver.experiment.to_pylist() if r["experiment_id"] == first["experiment_id"])
bits = next(iter_b8_records(first["detector_bits"], bits_per_record=exp_row["detector_count"]))
print("detector_event_count matches bit-by-bit count:", sum(bits) == first["detector_event_count"])

shots read 250,000 = accepted 250,000 + rejected 0: True
experiment.shots sums to shots read: True
trace covers every Silver row: True
trace rows per shot: 2000000 == 2000000
detector_event_count matches bit-by-bit count: True


In [11]:
# second run on unchanged input must give identical records and identifiers
again = build_google_silver(zf, bronze_object=google_object, input_sha256=google_sha256, run_id="notebook-exploration")
print("experiment identical:", silver.experiment.equals(again.experiment))
print("shot identical:      ", silver.shot.equals(again.shot))
print("trace identical:     ", silver.trace.equals(again.trace))

experiment identical: True
shot identical:       True
trace identical:      True


## 9. Break the input on purpose (seeds for the pytest tests)

Each case rebuilds a one-experiment copy of the archive with one flaw and checks the checks fire. The last case is the only one that must **not** stop the run.

In [12]:
FIRST = experiment_dirs(zf.namelist())[0]


def one_experiment_archive(replace=None):
    replace = replace or {}
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w") as out:
        for member in zf.namelist():
            if member.startswith(FIRST + "/"):
                out.writestr(member, replace.get(member, zf.read(member)))
    return zipfile.ZipFile(io.BytesIO(buffer.getvalue()))


def build(archive):
    return build_google_silver(archive, bronze_object=google_object, input_sha256=google_sha256, run_id="negative-test")


def expect_stop(label, replace):
    try:
        build(one_experiment_archive(replace))
    except GoogleSilverError as error:
        print(f"OK  {label}: stopped -> {error}")
    else:
        print(f"BAD {label}: did NOT stop")


measurements = zf.read(f"{FIRST}/measurements.b8")
actual = zf.read(f"{FIRST}/obs_flips_actual.01")
predicted = zf.read(f"{FIRST}/obs_flips_predicted_by_pymatching.01")
properties = zf.read(f"{FIRST}/properties.yml")

expect_stop("truncated measurements.b8", {f"{FIRST}/measurements.b8": measurements[:-1]})
expect_stop("actual .01 missing a line", {f"{FIRST}/obs_flips_actual.01": actual[:-2]})
expect_stop("predicted .01 has a non-binary value", {f"{FIRST}/obs_flips_predicted_by_pymatching.01": b"2" + predicted[1:]})
expect_stop("properties distance disagrees with the name", {f"{FIRST}/properties.yml": properties.replace(b"distance: 3", b"distance: 5")})
expect_stop("properties.yml missing a key", {f"{FIRST}/properties.yml": properties.replace(b"circuit_detectors", b"circuit_detectorz")})

# one shot with a non-zero padding bit: excluded + one issue row, everything else kept
dirty = bytearray(measurements)
dirty[27 * 5 + 26] |= 0x80   # last byte of shot 5 (27 bytes per shot for 209 bits)
partial = build(one_experiment_archive({f"{FIRST}/measurements.b8": bytes(dirty)}))
print(f"OK  dirty padding: accepted {partial.shot.num_rows:,} of {partial.shots_read:,}, issues: {len(partial.issues)}")
print(partial.issues[0])

OK  truncated measurements.b8: stopped -> surface_code_bX_d3_r25_center_3_5/measurements.b8: 1349999 bytes, expected 1350000 (50000 shots x 27 bytes for 209 bits)
OK  actual .01 missing a line: stopped -> surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01: 49999 lines, expected 50000
OK  predicted .01 has a non-binary value: stopped -> surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_pymatching.01: 01 data contains a value other than 0 or 1
OK  properties distance disagrees with the name: stopped -> surface_code_bX_d3_r25_center_3_5: name says (basis, distance, rounds, row, col)=('X', 3, 25, 3, 5) but properties.yml says ('X', 5, 25, 3, 5)
OK  properties.yml missing a key: stopped -> surface_code_bX_d3_r25_center_3_5/properties.yml is missing key(s): ['circuit_detectors']
OK  dirty padding: accepted 49,999 of 50,000, issues: 1
{'issue_id': 'dbe4716ffe5f3f767e7e9937da5155d07a0efbb49871849438732753d7202759', 'run_id': 'negative-test', 'source_record_id': 'bb3013cc7736c2853b67e

## Next steps

1. Agree the `source_record_id` scheme and the output location with the team (P5 owns tracing).
2. Move the function cells (2-6) into `prepare_data.py` or a Google module it calls; keep the notebook as the demo.
3. Turn section 9 into pytest tests on small in-memory archives, in the style of `tests/test_register_sources.py`.
4. Write `experiment`, `shot`, `trace` and `issues` to their target files and add the counts to `row_counts.json` / `run.json`.